# 01 Wilson-Cowan Criticality Sweep


In [ ]:
import numpy as np
import pandas as pd
from scipy.integrate import solve_ivp


def default_wc_params():
    return {
        "tau": 8.0,
        "r_e": 1.0,
        "r_i": 1.0,
        "c1": 16.0,
        "c2": 12.0,
        "c3": 15.0,
        "c4": 3.0,
        "c5": 0.0,
        "c6": 0.0,
        "aE": 1.3,
        "thetaE": 4.0,
        "aI": 2.0,
        "thetaI": 3.7,
    }


def sigmoid(x, a, theta):
    z = np.clip(-a * (x - theta), -500, 500)
    return 1.0 / (1.0 + np.exp(z)) - 1.0 / (1.0 + np.exp(a * theta))


def wilson_cowan_rhs(t, y, W, params):
    N = W.shape[0]
    E = y[:N]
    I = y[N:]

    tau = params["tau"]
    c1, c2, c3 = params["c1"], params["c2"], params["c3"]
    c4, c5, c6 = params["c4"], params["c5"], params["c6"]
    r_e, r_i = params["r_e"], params["r_i"]
    aE, thetaE = params["aE"], params["thetaE"]
    aI, thetaI = params["aI"], params["thetaI"]

    Smax_e = 1.0 - 1.0 / (1.0 + np.exp(aE * thetaE))
    Smax_i = 1.0 - 1.0 / (1.0 + np.exp(aI * thetaI))

    hE = c1 * E - c2 * I + c5 * (W @ E)
    hI = c3 * E - c4 * I + c6 * (W @ I)

    dE = (-E + (2.0 * Smax_e - r_e * E) * sigmoid(hE, aE, thetaE)) / tau
    dI = (-I + (2.0 * Smax_i - r_i * I) * sigmoid(hI, aI, thetaI)) / tau
    return np.concatenate([dE, dI])


def simulate_wc(W, params, T=150.0, dt=0.5, E0=0.1, I0=0.1):
    N = W.shape[0]
    y0 = np.concatenate([E0 * np.ones(N), I0 * np.ones(N)])
    t_eval = np.arange(0.0, T + dt, dt)

    sol = solve_ivp(
        lambda t, y: wilson_cowan_rhs(t, y, W, params),
        t_span=(0.0, T),
        y0=y0,
        t_eval=t_eval,
        method="RK45",
        rtol=1e-6,
        atol=1e-8,
    )
    if not sol.success:
        raise RuntimeError(sol.message)
    return sol.t, sol.y[:N], sol.y[N:]


def post_transient_activity(E, transient_fraction=0.5):
    start = int(transient_fraction * E.shape[1])
    return E[:, start:].mean(axis=1)


def sweep_c5(W, c5_values, T=150.0, dt=0.5):
    summary_rows = []
    regional_rows = []

    for c5 in c5_values:
        params = default_wc_params()
        params["c5"] = float(c5)
        params["c6"] = float(c5) / 4.0

        t, E, I = simulate_wc(W, params, T=T, dt=dt)
        regional_mean = post_transient_activity(E)

        summary_rows.append({
            "c5": float(c5),
            "c6": params["c6"],
            "mean_activity": float(regional_mean.mean()),
            "max_activity": float(regional_mean.max()),
        })
        regional_rows.append({
            "c5": float(c5),
            "c6": params["c6"],
            **{f"region_{i}": float(value) for i, value in enumerate(regional_mean)},
        })

    return pd.DataFrame(summary_rows), pd.DataFrame(regional_rows)


def estimate_critical_point(regional_df):
    c5 = regional_df["c5"].to_numpy(dtype=float)
    region_cols = [col for col in regional_df.columns if col.startswith("region_")]
    R = regional_df[region_cols].to_numpy(dtype=float)

    jump_scores = np.linalg.norm(np.diff(R, axis=0), axis=1)
    jump_index = int(np.argmax(jump_scores))
    return {
        "c5_pre": float(c5[jump_index]),
        "c5_star": float(c5[jump_index + 1]),
        "jump_size": float(jump_scores[jump_index]),
    }


# Example full call when W is available:
# c5_grid = np.linspace(0, 20, 101)
# summary_df, regional_df = sweep_c5(W, c5_grid)
# criticality = estimate_critical_point(regional_df)


## Plots

![Criticality sweep](./plots/criticality/105115_scale33_criticality_plot.png)

![Below transition](./plots/criticality/105115_scale33_timeseries_below_c5star_9.2.png)

![At transition](./plots/criticality/105115_scale33_timeseries_at_c5star_9.4.png)
